# Quantum Gate Playground

**Day 24 | 30-Day Quantum Computing Challenge**

## Aim:
To experiment with X, Y, Z, H, S, and T gates and compare their effects on quantum states.

## Tools:
Python, NumPy, Matplotlib (optional Qiskit)

## Expected Result:
Gate operations change basis states and/or phase according to their unitary transformations.

**Execution:** Run the cells from top to bottom. The notebook uses an ideal simulator unless explicitly stated otherwise.

## 1. Prepare gates and helper functions
The gates are unitary $2\times2$ matrices. A state is a complex vector $[\alpha,\beta]^T$ with $|\alpha|^2+|\beta|^2=1$.

In [ ]:
# Experiment step: execute this cell and inspect its output.
import numpy as np
import matplotlib.pyplot as plt
zero = np.array([1,0],dtype=complex)
X = np.array([[0,1],[1,0]],dtype=complex)
Y = np.array([[0,-1j],[1j,0]],dtype=complex)
Z = np.diag([1,-1]).astype(complex)
H = np.array([[1,1],[1,-1]],dtype=complex)/np.sqrt(2)
S = np.diag([1,1j]); T = np.diag([1,np.exp(1j*np.pi/4)])
gates = {'X':X,'Y':Y,'Z':Z,'H':H,'S':S,'T':T}
def probabilities(state): return np.abs(state)**2
def bloch(state):
    a,b=state
    return np.array([2*np.real(np.conj(a)*b),2*np.imag(np.conj(a)*b),abs(a)**2-abs(b)**2])
for name,g in gates.items():
    assert np.allclose(g.conj().T@g,np.eye(2)),f'{name} is not unitary'
print('All six gates are unitary.')

## 2. Gate-by-gate comparison from |0⟩
**Z, S and T leave |0⟩ unchanged.** This does not mean they are useless: their relative-phase effects are visible on superpositions.

In [ ]:
# Experiment step: execute this cell and inspect its output.
for name,g in gates.items():
    out=g@zero
    print(f'{name}: state={np.round(out,3)}, probabilities={np.round(probabilities(out),3)}, Bloch={np.round(bloch(out),3)}')

## 3. Repeat from |+⟩ to expose phase effects
Start with $|+\rangle=H|0\rangle$. S rotates its relative phase by $\pi/2$ and T by $\pi/4$. Both still give 50/50 outcomes in the Z basis.

In [ ]:
# Experiment step: execute this cell and inspect its output.
plus=H@zero
states_zero={name:g@zero for name,g in gates.items()}
states_plus={name:g@plus for name,g in gates.items()}
fig,axs=plt.subplots(1,2,figsize=(11,4))
for ax,states,title in [(axs[0],states_zero,'From |0>'),(axs[1],states_plus,'From |+>')]:
    ax.bar(list(states),[probabilities(s)[1] for s in states.values()])
    ax.set(ylim=(0,1.05),ylabel='P(1)',title=title)
plt.tight_layout();plt.savefig('results.png',dpi=180);plt.show()

## 4. Bloch-sphere comparison from |+⟩

In [ ]:
# Experiment step: execute this cell and inspect its output.
fig=plt.figure(figsize=(12,7))
for i,(name,state) in enumerate(states_plus.items(),1):
    ax=fig.add_subplot(2,3,i,projection='3d')
    u=np.linspace(0,2*np.pi,28);v=np.linspace(0,np.pi,16)
    ax.plot_wireframe(np.outer(np.cos(u),np.sin(v)),np.outer(np.sin(u),np.sin(v)),
        np.outer(np.ones_like(u),np.cos(v)),color='gray',alpha=.2,linewidth=.5)
    ax.quiver(0,0,0,*bloch(state),color='royalblue',linewidth=2.5,arrow_length_ratio=.15)
    ax.set(xlim=(-1,1),ylim=(-1,1),zlim=(-1,1),title=name+'|+>')
    ax.view_init(elev=23,azim=38)
plt.tight_layout();plt.savefig('bloch_states.png',dpi=180);plt.show()

## 5. Make your own gate sequence
Gates execute **left to right in the sequence string**. For example `HS` means apply H first, then S, so the matrix product is S@H.

In [ ]:
# Experiment step: execute this cell and inspect its output.
sequence='HST'  # edit, e.g. 'HH', 'HZ', 'HT', 'HSH'
state=zero.copy()
for letter in sequence:
    if letter not in gates: raise ValueError(f'Unknown gate: {letter}')
    state=gates[letter]@state
print('Sequence:',sequence,'state:',np.round(state,4))
print('P(0), P(1):',np.round(probabilities(state),4),'Bloch:',np.round(bloch(state),4))

## 6. Optional Qiskit circuit verification

In [ ]:
# Experiment step: execute this cell and inspect its output.
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    qc=QuantumCircuit(1)
    for letter in sequence: getattr(qc,letter.lower())(0)
    print(qc.draw(output='text'))
    print('Qiskit:',np.round(Statevector.from_instruction(qc).data,4))
    assert np.allclose(Statevector.from_instruction(qc).data,state)
except ImportError:
    print('Optional dependency missing: pip install qiskit')

## 7. Takeaways
- X flips computational-basis values; Y flips with a phase; Z flips the |1⟩ sign.
- H maps |0⟩ to |+⟩. S and T change relative phase.
- Gates with identical Z-basis probabilities can produce different quantum states.
- This is an **ideal statevector simulation**, not physical quantum hardware.

**SAMYAK DEKATE**

## Observation:
Record the measured or simulated outputs and compare them with the theoretical expectation. Note that finite-shot results can fluctuate.

## Conclusion:
The experiment demonstrates the quantum circuit or state transformation described above using simulation; it does not establish performance on physical quantum hardware.